<a href="https://colab.research.google.com/github/G0rav/machine_learning_explained_visually_free/blob/main/03-gradient-descent/notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Gradient Descent: How Machine Learning Models Actually Get Trained

**[Watch the video](https://youtu.be/LHj_JsDbesg)** · video 3 of *Machine Learning Explained
Visually*.

Everything from the video, runnable. Three layers:

1. **Follow along** — every run in the video, in code, in the same order.
2. **Experiment** — break it on purpose and find out exactly where it breaks.
3. **Challenge** — write the pieces yourself before the next video.

Nothing here imports from anywhere. `numpy` is required; `matplotlib` is
optional and every cell that uses it says so. Run it top to bottom.

Every section asserts the number the video put on screen. If a check ever
fails, the video and this notebook have drifted apart and one of them is
wrong.

---

# Layer 1 — Follow along

In [ ]:
import numpy as np

PASSED = 0
def check(label, got, want, tol=1e-9):
    """Assert a value the video states, and say so."""
    global PASSED
    got_f, want_f = float(got), float(want)
    assert abs(got_f - want_f) <= tol, f"{label}: {got_f!r} != {want_f!r}"
    PASSED += 1
    print(f"ok   {label:46s} {got_f:.9g}")

## Part 1 — Why we iterate

Video 2 finished with a method: differentiate, set the derivative to zero,
solve for `x`. On `f(x) = x^2 - 3x + 2` that works and takes one line.

In [ ]:
def f(x):      return x ** 2 - 3 * x + 2
def df(x):     return 2 * x - 3

# 2x - 3 = 0  =>  x = 1.5. Solved, not searched.
X_STAR = 3 / 2
check("part 1  the minimum, solved exactly", X_STAR, 1.5)
check("part 1  f at the minimum", f(X_STAR), -0.25)
check("part 1  the derivative there", df(X_STAR), 0.0)

### The one that will not solve

The regularised logistic loss from video 2, in one variable. Differentiate it
and set that to zero and you get `0.2x = 1/(1 + e^x)` — `x` alone on one side
and inside an exponential on the other, with no rearrangement that frees it.

There is nothing wrong with the *answer*: the two sides cross exactly once, so
it plainly exists. What there is no formula for is writing it down.

In [ ]:
LAM = 0.1

def g(x):      return np.log(1 + np.exp(-x)) + LAM * x ** 2
def dg(x):     return 2 * LAM * x - 1 / (1 + np.exp(x))

lhs = lambda x: 2 * LAM * x          # the 0.2x side
rhs = lambda x: 1 / (1 + np.exp(x))  # the 1/(1+e^x) side

# One crossing: lhs - rhs changes sign once over any window you like.
grid = np.linspace(0.0, 3.0, 4001)
crossings = int(np.sum(np.diff(np.sign(lhs(grid) - rhs(grid))) != 0))
check("part 1  the stationary equation has one root", crossings, 1)

# And it is not a number you can write a formula for -- only one you can find.
root = grid[np.argmin(np.abs(lhs(grid) - rhs(grid)))]
print(f"     the root is near {root:.4f}; part 8 reaches it without solving")

### Minimising is maximising, upside down

Worth a line because it halves the work: the `x` that minimises `f` is the `x`
that maximises `-f`. Negating flips every value, so the smallest becomes the
largest — and `x` itself does not move.

In [ ]:
xs = np.linspace(-1.0, 4.0, 5001)
check("part 1  argmin f == argmax -f",
      xs[np.argmin(f(xs))], xs[np.argmax(-f(xs))])
print("     so everything from here is written for minima only")

## Part 2 — The sign around a minimum

The whole algorithm runs on one fact. To the right of a minimum the derivative
is positive; to the left it is negative; at the minimum it is zero. **The sign
already says which side you are on**, which is why no step of this ever asks.

In [ ]:
check("part 2  right of the minimum, x = 3", df(3.0), 3.0)
check("part 2  at the minimum,      x = 1.5", df(1.5), 0.0)
check("part 2  left of the minimum, x = 0", df(0.0), -3.0)

for x in (3.0, 1.5, 0.0):
    side = "right of" if x > X_STAR else ("at" if x == X_STAR else "left of")
    print(f"     x = {x:4.1f}  f'(x) = {df(x):+5.1f}   {side} the minimum")

### And the size shrinks as you come in

The second thing, which part 5 turns into the most useful property the
algorithm has: `|f'|` gets smaller the closer you are. It is `3`, `1.2`,
`0.48` coming in from the right, and the same three sizes coming in from the
left with the sign the other way round.

In [ ]:
for x in (3.0, 2.1, 1.74):
    check(f"part 2  |f'| at x = {x}", abs(df(x)), abs(2 * x - 3))
for x, want in ((3.0, 3.0), (2.1, 1.2), (1.74, 0.48)):
    check(f"part 2  from the right, x = {x}", abs(df(x)), want, 1e-12)
for x, want in ((0.0, 3.0), (0.9, 1.2), (1.26, 0.48)):
    check(f"part 2  from the left,  x = {x}", abs(df(x)), want, 1e-12)

# |f'(x)| = 2|x - 1.5| exactly: the derivative IS the distance, doubled.
probe = np.linspace(-2.0, 5.0, 701)
check("part 2  |f'(x)| == 2|x - x*| everywhere",
      np.max(np.abs(np.abs(df(probe)) - 2 * np.abs(probe - X_STAR))), 0.0, 1e-12)

## Part 3 — One step

The rule. `r` is a constant you choose, always positive; the video uses `0.3`
and does not justify it until part 6.

$$x_1 = x_0 - r\,\frac{df}{dx}\Big|_{x_0}$$

In [ ]:
R = 0.3

def step(x, r=R):
    """One update. No branch, no test for which side of the minimum x is on."""
    return x - r * df(x)

# From the right: 3 -> 2.1. The derivative is +3, so we subtract 0.9.
check("part 3  from x0 = 3, the derivative", df(3.0), 3.0)
check("part 3  from x0 = 3, the step", R * df(3.0), 0.9)
check("part 3  from x0 = 3, x1", step(3.0), 2.1)

# From the left: 0 -> 0.9. The derivative is -3, and minus a minus is a plus.
check("part 3  from x0 = 0, the derivative", df(0.0), -3.0)
check("part 3  from x0 = 0, x1", step(0.0), 0.9)

# Further out still, and it still holds: -1 -> 0.5.
check("part 3  from x0 = -1, the derivative", df(-1.0), -5.0)
check("part 3  from x0 = -1, x1", step(-1.0), 0.5)

**That is the part.** The same expression, with the same sign in it, moved us
left from the right-hand side and right from the left-hand side. We never
checked which side we were on, and there is no `if` anywhere — the derivative
already carries that information, in its sign.

In [ ]:
for x0 in (3.0, 0.0, -1.0):
    x1 = step(x0)
    closer = abs(x1 - X_STAR) < abs(x0 - X_STAR)
    assert closer, f"{x0} did not move closer"
    print(f"     x0 = {x0:5.1f} -> x1 = {x1:6.3f}   "
          f"gap {abs(x0 - X_STAR):.3f} -> {abs(x1 - X_STAR):.3f}   closer")
PASSED += 1

## Part 4 — The loop

Same rule with an index on it, run until it stops moving.

$$x_{i+1} = x_i - r\,\frac{df}{dx}\Big|_{x_i}$$

In [ ]:
def descend(x0, r=R, steps=12):
    """The whole algorithm. Returns every x it held, starting with x0."""
    xs = [float(x0)]
    for _ in range(steps):
        xs.append(step(xs[-1], r))
    return np.array(xs)

trace = descend(0.0)
for k, want in enumerate((0.0, 0.9, 1.26, 1.404, 1.4616, 1.48464)):
    check(f"part 4  x{k}", trace[k], want, 1e-12)

### So when do we stop?

Mathematically it arrives and stays: once the derivative is zero the step is
zero and nothing moves again. A program cannot wait for that — **with real
arithmetic the derivative essentially never reaches exactly zero.** So the
test is on the movement instead, and how small counts as small is another
number you choose, called the tolerance.

In [ ]:
def stop_when_still(x0, r=R, tol=1e-6, cap=100_000):
    """Update until |x_{k+1} - x_k| < tol. Returns (updates, x)."""
    x = float(x0)
    for k in range(1, cap + 1):
        nxt = step(x, r)
        if abs(nxt - x) < tol:
            return k, nxt
        x = nxt
    raise RuntimeError("did not settle")

for tol, want_k, want_x in ((1e-4, 11, 1.49993708544),
                            (1e-6, 16, 1.4999993557549056),
                            (1e-9, 24, 1.4999999995777875)):
    k, x = stop_when_still(0.0, tol=tol)
    check(f"part 4  tol {tol:<6g} updates", k, want_k)
    check(f"part 4  tol {tol:<6g} answer", x, want_x, 1e-12)

print("     13 extra updates buys 5 extra digits -- that is the whole trade")

In [ ]:
# The four the video puts on screen. It is still not zero after forty.
long_run = descend(0.0, steps=40)
for k, want in ((11, -0.00012582911999992064),
                (16, -1.2884901887666445e-06),
                (24, -8.444249743888577e-10),
                (40, -4.440892098500626e-16)):
    check(f"part 4  f' after {k:>2} updates", df(long_run[k]), want, 1e-18)
print("     -1.3e-04 down to -4.4e-16, and none of them zero -- so the test")
print("     is on the movement. Layer 2 looks at what happens after forty.")

## Part 5 — The step shrinks by itself

Look at the sizes of the steps that run took: `0.9`, `0.36`, `0.144`,
`0.0576`, `0.02304`. Each one smaller than the last — and **nobody wrote that
rule.** `r` was `0.3` the whole way.

In [ ]:
steps_taken = np.abs(np.diff(trace))
for k, want in enumerate((0.9, 0.36, 0.144, 0.0576, 0.02304)):
    check(f"part 5  step {k}", steps_taken[k], want, 1e-12)

### Why, and by exactly how much

The step is `r` times the derivative and `r` is fixed, so the only thing that
can move is the derivative — which part 2 showed shrinks as you approach.

For this function you can say exactly how much. `|f'(x)| = 2|x - 1.5|`, so the
derivative *is* the distance to the minimum, doubled. A step of `r * 2d`
therefore closes `2r` of that distance and leaves `1 - 2r` of it — which at
`r = 0.3` is `0.4`, every single time.

In [ ]:
RATIO = 1 - 2 * R
check("part 5  the ratio 1 - 2r", RATIO, 0.4)

ratios = steps_taken[1:] / steps_taken[:-1]
check("part 5  every measured ratio is 1 - 2r",
      float(np.max(np.abs(ratios - RATIO))), 0.0, 1e-12)

gaps = np.abs(trace - X_STAR)
check("part 5  the gap shrinks by the same factor",
      float(np.max(np.abs(gaps[1:] / gaps[:-1] - RATIO))), 0.0, 1e-12)

print("     far out: a long step. close in: a short one. From one constant.")

## Part 6 — The learning rate

`r` is the *learning rate*, also called the step size; in papers it is usually
the Greek letter eta. Nothing in the algorithm computes it — you supply it
before the loop starts. That makes it a **hyperparameter**, and picking it is
the thing that decides whether any of this works.

### First way to get it wrong: too small

Same function, same start, `r = 0.02` instead of `0.3`. Every single step goes
the right way. It is never confused and never goes backwards. It is just slow.

In [ ]:
R_SMALL = 0.02
small = descend(0.0, r=R_SMALL, steps=500)
for k, want in ((1, 0.06), (5, 0.2769412), (25, 0.9594047), (100, 1.4746954)):
    check(f"part 6  r = 0.02, after {k:>3} updates", small[k], want, 1e-6)
check("part 6  r = 0.02, after 500 updates", small[500], 1.5, 1e-6)

# Every step went the right way -- nothing was wrong, it was only slow.
assert np.all(np.diff(small) >= 0), "a step went the wrong way"

drop = np.diff(f(small))
falls_for = int(np.argmax(drop >= 0))
check("part 6  updates the loss falls for, uninterrupted", falls_for, 410)
assert np.all(drop[:falls_for] < 0), "the loss went up before the floor"

# After that it is at the resolution of a float near -0.25 and wobbles there
# by about 9e-16. That is not the algorithm changing its mind, it is the
# last two bits of a double, and it is why "the loss stopped improving" is a
# test you write with a tolerance and not with `<`.
check("part 6  and then bounces by no more than", float(drop.max()),
      8.881784197001252e-16, 1e-18)
print(f"     410 updates of honest descent, then float noise at 9e-16")

**And that is why too small is the dangerous one.** Watch the loss and you see
a number that is still falling — which looks exactly like a run that is
working, because it *is* a run that is working. What you cannot see is whether
it needs another hundred updates or another ten million.

In [ ]:
# The window the video draws is the first hundred updates -- which is about
# as long as anyone watches before deciding whether a run is healthy.
WINDOW = 100
loss = f(small)[:WINDOW + 1]
still_falling = bool(loss[-1] < loss[-2] < loss[-3])
check("part 6  still falling at the end of the window", still_falling, True)
print(f"     last four losses in the window: "
      f"{', '.join(f'{v:.6f}' for v in loss[-4:])}")
print("     down, and down, and down. Nothing here tells you it needs 410.")

Compare the healthy rate over the same window. `r = 0.3` is at its floor by
the sixth update and the rest of the picture is a horizontal line; `r = 0.02`
is a line still on its way down. **Still falling looks like progress and flat
looks like finished, and both readings are backwards.**

In [ ]:
def updates_to_within(r, window=0.01, x0=0.0, cap=100_000):
    # How many updates until |x - x*| first drops below `window`.
    x = float(x0)
    for k in range(cap + 1):
        if abs(x - X_STAR) < window:
            return k
        x = step(x, r)
    raise ValueError(f"never got within {window} at r={r}")

check("part 6  r = 0.3  updates to get within 0.01", updates_to_within(R), 6)
check("part 6  r = 0.02 updates to get within 0.01",
      updates_to_within(R_SMALL), 123)
print("     six against a hundred and twenty three, for the same accuracy")
print("     -- and on a real model an update is minutes, not microseconds")

## Part 7 — When it never arrives

### Second way: too large

At `r = 1` on this function the step is exactly twice the distance to the
minimum, so it lands the same distance out the other side. Forever. The loss
is *constant*, which is the tell: a flat loss curve does not mean finished.

In [ ]:
osc = descend(0.0, r=1.0, steps=8)
check("part 7  r = 1, x1", osc[1], 3.0)
check("part 7  r = 1, x2", osc[2], 0.0)
check("part 7  r = 1, x3", osc[3], 3.0)
check("part 7  r = 1, the loss never moves", float(np.ptp(f(osc))), 0.0, 1e-12)
check("part 7  r = 1, and the value it sticks at", f(osc[0]), 2.0)
print("     0, 3, 0, 3, ... and |x_{k+1} - x_k| = 3 every time, so no tolerance fires")

Push `r` a little past that and it does not merely fail to arrive — it leaves.

In [ ]:
div = descend(0.0, r=1.05, steps=6)
for k, want in ((1, 3.15), (2, -0.315), (3, 3.4965), (4, -0.69615)):
    check(f"part 7  r = 1.05, x{k}", div[k], want, 1e-9)

moves = np.abs(np.diff(div))
assert np.all(np.diff(moves) > 0), "the steps should be growing"
print(f"     the steps grow: {', '.join(f'{m:.4f}' for m in moves[:5])}")
print("     a max-iteration cap is not belt and braces, it is the only exit")
PASSED += 1

### The fix: stop keeping `r` constant

Make it a function of the iteration number — `r = h(i)`, anything that comes
down as `i` goes up. The oscillation happened because the step was big enough
to cross and land the same distance out; shrink `r` and each step lands nearer
than the last.

In [ ]:
def descend_scheduled(x0, r0=1.0, gamma=0.6, steps=6):
    """r = r0 * gamma**i -- the simplest schedule that works."""
    x, xs = float(x0), [float(x0)]
    for i in range(steps):
        x = x - (r0 * gamma ** i) * df(x)
        xs.append(x)
    return np.array(xs)

sched = descend_scheduled(0.0)
sched_gap = np.abs(sched - X_STAR)
osc_gap = np.abs(descend(0.0, r=1.0, steps=6) - X_STAR)

# It starts at the rate that was oscillating, so the FIRST step is the same
# step the failing run made -- 0 to 3. The two part company from the second.
check("part 7  the schedule's first step matches the failure",
      sched[1], 3.0)
assert np.all(np.diff(sched_gap[1:]) < 0), "it stopped getting closer"

print(f"     {'i':>2}  {'r_i':>7}  {'scheduled gap':>14}  {'constant r = 1':>15}")
for i in range(len(sched)):
    r_i = "" if i == 0 else f"{1.0 * 0.6 ** (i - 1):7.4f}"
    print(f"     {i:>2}  {r_i:>7}  {sched_gap[i]:14.5f}  {osc_gap[i]:15.5f}")
print("\n     one column comes down after the first step; the other never does")

## Part 8 — Vectors, and a function we cannot solve

### One substitution

In machine learning `x` is a vector with `d` components, and the only thing
that changes is that `df/dx` becomes `grad f` — the vector of partial
derivatives, one per component.

$$x_{i+1} = x_i - r\,\nabla f(x_i)$$

Same rule. The subtraction just happens component by component, and the
algorithm does not know or care that there is more than one dimension.

In [ ]:
def grad_numeric(fn, v, h=1e-6):
    """grad, one partial derivative at a time. Checks the algebra, not fast."""
    v = np.asarray(v, float)
    out = np.zeros_like(v)
    for j in range(v.size):
        up, dn = v.copy(), v.copy()
        up[j] += h; dn[j] -= h
        out[j] = (fn(up) - fn(dn)) / (2 * h)
    return out

def bowl(v):    return float(np.sum(v ** 2) - 3 * np.sum(v) + 2 * v.size)

start = np.array([3.0, 0.0, -1.0])   # the three starts part 3 used, at once
v = start.copy()
first = v - R * grad_numeric(bowl, v)
# The first update IS part 3's three runs, side by side and in one line.
check("part 8  component 0 after one step", first[0], 2.1, 1e-5)
check("part 8  component 1 after one step", first[1], 0.9, 1e-5)
check("part 8  component 2 after one step", first[2], 0.5, 1e-5)

v = start.copy()
for _ in range(25):
    v = v - R * grad_numeric(bowl, v)
check("part 8  vector run, component 0", v[0], X_STAR, 1e-6)
check("part 8  vector run, component 1", v[1], X_STAR, 1e-6)
check("part 8  vector run, component 2", v[2], X_STAR, 1e-6)
print(f"     started {start}, landed {np.round(v, 6)} -- nothing else changed")

### And now one that will not solve

Everything so far ran on `x^2 - 3x + 2`, chosen because video 2 solved it
exactly — which means gradient descent has not yet been asked to do anything
algebra could not. So: the function from part 1, the one with no closed form.

$$f(x) = \log\!\left(1 + e^{-x}\right) + 0.1x^2
\qquad
\frac{df}{dx} = 0.2x - \frac{1}{1 + e^{x}}$$

In [ ]:
R8, TOL8 = 2.0, 1e-6

xs8 = [0.0]
for _ in range(5):
    xs8.append(xs8[-1] - R8 * dg(xs8[-1]))
for k, want in ((1, 1.0), (2, 1.13788284), (3, 1.16814828), (5, 1.17696914)):
    check(f"part 8  logistic run, x{k}", xs8[k], want, 1e-8)

In [ ]:
# x8 and k8, not x and k: Layer 2 runs its own loops over short names, and a
# reader who jumps around should not have part 8's answer quietly replaced.
x8, k8 = 0.0, 0
while True:
    k8 += 1
    nxt = x8 - R8 * dg(x8)
    if abs(nxt - x8) < TOL8:
        x8 = nxt
        break
    x8 = nxt

check("part 8  updates to settle at tol 1e-6", k8, 11)
check("part 8  and where it landed", x8, 1.1775051619335128, 1e-12)

# It really is the minimum: the derivative there is ~0 and both neighbours are higher.
check("part 8  the derivative there is ~0", dg(x8), 0.0, 1e-6)
assert g(x8) < g(x8 - 1e-3) and g(x8) < g(x8 + 1e-3)
print(f"     eleven updates, on a function whose minimum cannot be written down")
PASSED += 1

In [ ]:
print(f"Layer 1 complete — {PASSED} checks passed, all of them numbers the video states.")

---

# Layer 2 — Experiment

The video shows `r = 0.3` working, `r = 0.02` crawling, `r = 1` oscillating
and `r = 1.05` leaving. Those are four points on a line. Here is the whole
line.

## Exactly where the learning rate stops working

Part 5 derived that each step leaves `1 - 2r` of the distance. That number is
also the answer to "does this converge?" — the gap is multiplied by `|1 - 2r|`
every update, so it shrinks only while `|1 - 2r| < 1`, which is `0 < r < 1`.
The boundary is not a rule of thumb; it is arithmetic.

In [ ]:
print(f"{'r':>6}  {'|1-2r|':>8}  {'gap after 40':>14}   verdict")
for r in (0.02, 0.1, 0.3, 0.5, 0.7, 0.9, 0.99, 1.0, 1.05, 1.5):
    run = descend(0.0, r=r, steps=40)
    gap = abs(run[-1] - X_STAR)
    factor = abs(1 - 2 * r)
    if factor < 1:      verdict = "converges"
    elif factor == 1:   verdict = "oscillates forever"
    else:               verdict = "diverges"
    shown = f"{gap:14.6g}" if np.isfinite(gap) else f"{'overflow':>14}"
    print(f"{r:6.2f}  {factor:8.2f}  {shown}   {verdict}")

print("\nr = 0.5 is the interesting one: 1 - 2r = 0, so it lands on the answer")
print("in a single step. That is special to this curve, not a general trick.")

In [ ]:
one_step = descend(0.0, r=0.5, steps=1)
assert abs(one_step[1] - X_STAR) < 1e-15
print(f"     x0 = 0 -> x1 = {one_step[1]}  (exactly x*, in one update)")

## What a tolerance actually buys

Part 4 quotes three tolerances. Here is the whole curve, and the thing worth
noticing is that the cost is *logarithmic* — each extra digit of accuracy
costs a roughly constant number of updates, because the gap shrinks by a
constant factor every time.

In [ ]:
print(f"{'tolerance':>12}  {'updates':>8}  {'error':>12}")
prev = None
for e in range(2, 13):
    tol = 10.0 ** -e
    k, x = stop_when_still(0.0, tol=tol)
    delta = "" if prev is None else f"   (+{k - prev})"
    print(f"{tol:12.0e}  {k:8d}  {abs(x - X_STAR):12.3e}{delta}")
    prev = k
print("\nA constant number of extra updates per decimal place, which is what")
print("'13 extra updates for 5 extra digits' is a sample of.")

## The floor, and why you still do not test the derivative

Part 4 says a program cannot wait for the derivative to reach exactly zero,
*because with real arithmetic it usually never will*. That "usually" is doing
real work, and this is the cell that shows why.

Run past the forty updates the video draws and this particular curve, at this
particular rate, **does** land exactly on `1.5` — the update is
`x <- 0.4x + 0.9`, and once `x` is the nearest float to `1.5` that map returns
it unchanged. So the derivative really does become `0.0`.

That is luck, not a guarantee. It happens here because `1.5` is exactly
representable and the arithmetic happens to land on it; move the minimum to
`1/3` and it never will. A termination test you cannot justify on every curve
is not a termination test, which is why the movement is what gets measured.

In [ ]:
x, hit = 0.0, None
for k in range(1, 200):
    x = step(x)
    if df(x) == 0.0 and hit is None:
        hit = k
    if k in (10, 20, 40, 41, 60, 120):
        print(f"     after {k:>3} updates   x - 1.5 = {x - X_STAR:+.3e}   "
              f"f'(x) = {df(x):+.3e}")
print(f"\n     the derivative first reads exactly 0.0 at update {hit}")
print("     -- one past the last row the video shows")

So is the video wrong? No, and the reason is the point. **Change only the
learning rate and the same curve stops reaching zero at all.** Whether the
arithmetic happens to land on the fixed point depends on the curve, the
rate and the start, and there is no way to know which you have in advance.

A termination test you cannot justify before you run it is not a
termination test. That is why the test is on the movement.

In [ ]:
def reaches_zero(deriv, r, x0=0.0, cap=4000):
    """Does the derivative ever read exactly 0.0? The update, or None."""
    x = float(x0)
    for k in range(1, cap + 1):
        x = x - r * deriv(x)
        if not np.isfinite(x):
            return "diverged"
        if deriv(x) == 0.0:
            return k
    return None

print(f"{'curve':<26} {'r':>5}   exactly 0.0?")
for name, deriv, r in (("x^2 - 3x + 2", df, 0.3),
                       ("x^2 - 3x + 2", df, 0.1),
                       ("x^2 - 3x + 2", df, 0.7),
                       ("log(1+e^-x) + 0.1x^2", dg, 2.0),
                       ("log(1+e^-x) + 0.1x^2", dg, 0.5)):
    got = reaches_zero(deriv, r)
    if got is None:
        verdict = "never, in 4000 updates"
    elif isinstance(got, str):
        verdict = got
    else:
        verdict = f"yes, at update {got}"
    print(f"{name:<26} {r:>5}   {verdict}")

print("\n     Same curve, two rates, opposite answers -- and part 8's")
print("     function never gets there at all. You cannot tell which one")
print("     you have until you have already run it.")

## Optional: see it (needs `matplotlib`)

In [ ]:
try:
    import matplotlib.pyplot as plt

    grid = np.linspace(-0.6, 3.6, 400)
    fig, axes = plt.subplots(1, 3, figsize=(13, 3.6), sharey=True)
    for ax, r, title in zip(axes, (0.3, 1.0, 1.05),
                            ("r = 0.3, converges", "r = 1, oscillates",
                             "r = 1.05, diverges")):
        run = descend(0.0, r=r, steps=8)
        run = run[np.abs(run) < 6]
        ax.plot(grid, f(grid), lw=2)
        ax.plot(run, f(run), "o-", ms=5, lw=1)
        ax.axvline(X_STAR, ls="--", lw=1)
        ax.set_title(title)
        ax.set_xlabel("x")
    axes[0].set_ylabel("f(x)")
    fig.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not installed — skipping the picture, nothing else needs it")

---

# Layer 3 — Challenge

Four exercises, in the order the next video needs them. Each has a check you
can run against your own answer.

**1. Write the loop.** One function: start at `x0`, update until the move is
smaller than `tol`, return how many updates it took and where it landed. This
is the whole of parts 3 and 4 in about five lines.

In [ ]:
def my_descend(x0, r, tol):
    # replace with your own answer -- return (updates, x)
    return None

if my_descend(0.0, 0.3, 1e-6) is None:
    print("not attempted yet — edit my_descend above")
else:
    for r, tol in ((0.3, 1e-4), (0.3, 1e-6), (0.1, 1e-6)):
        got = my_descend(0.0, r, tol)
        want = stop_when_still(0.0, r=r, tol=tol)
        ok = got[0] == want[0] and abs(got[1] - want[1]) < 1e-12
        print(f"{'ok  ' if ok else 'FAIL'} r = {r}, tol = {tol:g}   "
              f"yours {got}   reference {want}")

**2. Predict before you run.** For `f(x) = x^2 - 3x + 2`, part 5 derived that
the distance to the minimum is multiplied by `1 - 2r` each update. Use that —
not a loop — to say how many updates it takes to get the gap below `tol` from
`x0`. Then check it against the loop.

In [ ]:
def my_updates_needed(x0, r, tol):
    # replace with your own answer -- an integer, from the formula, no loop
    return None

if my_updates_needed(0.0, 0.3, 1e-6) is None:
    print("not attempted yet — edit my_updates_needed above")
else:
    for r, tol in ((0.3, 1e-4), (0.3, 1e-6), (0.3, 1e-9), (0.1, 1e-6)):
        got = my_updates_needed(0.0, r, tol)
        want = stop_when_still(0.0, r=r, tol=tol)[0]
        # The loop tests the MOVE and you are predicting the GAP, so being
        # within one update is the right answer, not an approximate one.
        ok = abs(int(got) - want) <= 1
        print(f"{'ok  ' if ok else 'FAIL'} r = {r}, tol = {tol:g}   "
              f"yours {int(got):3d}   loop {want:3d}")

**3. Make `r = 1.05` converge.** It diverges on its own. Write a schedule —
any `r(i)` that comes down as `i` goes up — that gets it to the minimum
anyway. There is more than one right answer; the check only asks that you
arrive.

In [ ]:
def my_schedule(i):
    # replace with your own answer -- the learning rate to use at iteration i
    return None

if my_schedule(0) is None:
    print("not attempted yet — edit my_schedule above")
else:
    x = 0.0
    for i in range(80):
        x = x - my_schedule(i) * df(x)
    gap = abs(x - X_STAR)
    print(f"landed at {x:.8f}, gap {gap:.2e}")
    print("ok" if gap < 1e-4 else "not there yet — does your r actually decrease?")

**4. Before the next video.** Everything here minimised a function somebody
chose. The next video minimises a *loss over a dataset* — the same rule, with
the sum over your training rows inside the gradient.

Here are six cars: age in years, price in thousands. Fit `price = w * age + b`
by gradient descent on the squared error, and notice the one thing that
changes: the gradient is now a sum with one term per row, so **every update
reads the whole dataset.** That cost is what video 4 is about.

In [ ]:
AGE   = np.array([1.0, 2.0, 4.0, 6.0, 8.0, 9.0])
PRICE = np.array([23.2, 19.6, 15.2, 11.2, 7.6, 7.2])

def car_loss(w, b):
    return float(np.sum((PRICE - (w * AGE + b)) ** 2))

def my_fit(w0=-0.5, b0=18.0, r=0.002, steps=2000):
    # replace with your own answer -- return (w, b)
    return None

if my_fit() is None:
    print("not attempted yet — edit my_fit above")
    print("hint: dL/dw = sum(-2 * age * (price - (w*age + b))), and dL/db is")
    print("      the same without the age factor. One sum per row, per update.")
else:
    w, b = my_fit()
    print(f"yours   w = {w:.4f}, b = {b:.4f}   loss {car_loss(w, b):.4f}")
    print(f"target  w = -2, b = 24 is the exact answer for this data")
    print("ok" if abs(w + 2) < 0.1 and abs(b - 24) < 0.6 else "keep going")

---

That is video 3: one rule, applied over and over, with one number you have to
choose and two ways of choosing it badly.

**Next:** *Gradient Descent for Linear Regression* — the same algorithm
pointed at a real model, what one update actually costs when the sum runs over
every row you have, and the trick that makes it affordable.

**[Watch the video](https://www.youtube.com/@school_whool)**